In [1]:
import sys
import os
sys.path.append(os.path.abspath('/home/xilinx/jupyter_notebooks/EE712/Project/ml-kem'))

import numpy as np
import time
from mlkem.math.constants import n, q
from mlkem.math.field import Zm
from mlkem.math.polynomial_ring import PolynomialRing, RingRepresentation

from mlkem.auxiliary.ntt import ntt as sw_ntt, ntt_inv as sw_ntt_inv
from mlkem.auxiliary.ntt_hardware import ntt as hw_ntt, ntt_inv as hw_ntt_inv

Loading ML-KEM NTT Hardware Overlay...


Hardware loaded successfully!


In [2]:
print("Generating random polynomial for test...")
random_coeffs = [np.random.randint(0, q) for _ in range(n)]
poly_orig = PolynomialRing([Zm(c, q) for c in random_coeffs], RingRepresentation.STANDARD)
print("Polynomial generated.")

Generating random polynomial for test...
Polynomial generated.


In [9]:
print("--- Forward NTT Benchmark ---")

# Software Forward NTT (Deep copy to avoid in-place mutation of orig)
poly_sw_copy = PolynomialRing([Zm(c.val, q) for c in poly_orig.coefficients], RingRepresentation.STANDARD)
start_sw = time.time()
poly_ntt_sw = sw_ntt(poly_sw_copy)
end_sw = time.time()
time_sw = (end_sw - start_sw) * 1000
print(f"Software NTT completed in {time_sw:.4f} ms")

# Hardware Forward NTT (Deep copy just in case)
poly_hw_copy = PolynomialRing([Zm(c.val, q) for c in poly_orig.coefficients], RingRepresentation.STANDARD)
start_hw = time.time()
poly_ntt_hw = hw_ntt(poly_hw_copy)
end_hw = time.time()
time_hw = (end_hw - start_hw) * 1000
print(f"Hardware NTT completed in {time_hw:.4f} ms")

speedup_fwd = time_sw / time_hw if time_hw > 0 else 0
print(f"-> Forward NTT Speedup: {speedup_fwd:.2f}x")

--- Forward NTT Benchmark ---
Software NTT completed in 46.5865 ms
Hardware NTT completed in 19.5582 ms
-> Forward NTT Speedup: 2.38x


In [10]:
print("\n--- Inverse NTT Benchmark ---")

# Software Inverse NTT
# MUST deep copy because sw_ntt_inv mutates in place!
ntt_sw_copy = PolynomialRing([Zm(c.val, q) for c in poly_ntt_sw.coefficients], RingRepresentation.NTT)
start_sw = time.time()
poly_inv_sw = sw_ntt_inv(ntt_sw_copy)
end_sw = time.time()
time_sw = (end_sw - start_sw) * 1000
print(f"Software INTT completed in {time_sw:.4f} ms")

# Hardware Inverse NTT
start_hw = time.time()
poly_inv_hw = hw_ntt_inv(poly_ntt_hw)
end_hw = time.time()
time_hw = (end_hw - start_hw) * 1000
print(f"Hardware INTT completed in {time_hw:.4f} ms")

speedup_inv = time_sw / time_hw if time_hw > 0 else 0
print(f"-> Inverse NTT Speedup: {speedup_inv:.2f}x")


--- Inverse NTT Benchmark ---
Software INTT completed in 50.0691 ms
Hardware INTT completed in 20.1483 ms
-> Inverse NTT Speedup: 2.49x


In [11]:
print("\n--- Verifying Correctness ---")
errors_fwd = sum(1 for i in range(n) if poly_ntt_hw.coefficients[i].val != poly_ntt_sw.coefficients[i].val)
errors_inv = sum(1 for i in range(n) if poly_inv_hw.coefficients[i].val != poly_inv_sw.coefficients[i].val)
errors_orig = sum(1 for i in range(n) if poly_inv_hw.coefficients[i].val != poly_orig.coefficients[i].val)

if errors_fwd == 0 and errors_inv == 0 and errors_orig == 0:
    print("\n==================================================")
    print("✅ SUCCESS: Hardware perfectly matches Software!")
    print("==================================================")
else:
    print(f"\n❌ FAILURE:\n- Forward Mismatches: {errors_fwd}\n- Inverse Mismatches: {errors_inv}\n- Original Mismatches: {errors_orig}")


--- Verifying Correctness ---

✅ SUCCESS: Hardware perfectly matches Software!
